In [1]:
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path
import importlib.util
import logging
import traceback
from typing import Dict, List, Any, Optional, Tuple
from sklearn.preprocessing import MinMaxScaler # Explicit import
import warnings # For warning management

# --- Logging configuration for this module ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Filtering specific warnings ---
# Suppress scikit-learn validation warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.utils.validation')
# Suppress scikit-learn deprecation warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn.utils.deprecation')
# Suppress specific warnings from the dynamically loaded PFL module
warnings.filterwarnings('ignore', message="Unexpected GMM covariance structure (ndim=2). Calculating widths based on minimum centroid distances (fallback).", module='PFLZ_module_temp_simulateur')

# Declaration of a placeholder for the PFL_CP class
# This class will be overwritten by the dynamically loaded PFL module.
# It defines the expected attributes and methods to avoid type errors before the actual loading.
class PFLZ_CP_Placeholder:
    def __init__(self, n_clusters: int, epsilon: float = 1e-8, lgb_params: Optional[Dict[str, Any]] = None, feature_names: Optional[List[str]] = None) -> None:
        # The actual implementation of __init__ will come from the loaded PFL module.
        pass
    def calc_mf(self, X: np.ndarray, normalized: bool = False) -> np.ndarray:
        raise NotImplementedError("The calc_mf method has not been implemented by the loaded PFL module.")
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError("The predict_proba method has not been implemented by the loaded PFL module.")
    # Attributes that will be defined after the fit of the real PFL model
    cond_probs_: Optional[np.ndarray] = None
    cluster_centers_: np.ndarray = np.array([])
    feature_names: Optional[List[str]] = None
    n_clusters: int = 0
    n_original_features_: int = 0
    # Add other attributes if the simulator accesses them directly before the fit, e.g.:
    # lgb_model: Any = None # If we want to access the internal LightGBM model after the PFL fit
    def calc_rule_widths(self) -> np.ndarray:
        raise NotImplementedError("The calc_rule_widths method has not been implemented by the loaded PFL module.")


PFLZ_CP = PFLZ_CP_Placeholder # Initialization of the global placeholder
_PFLZ_MODULE_LOADED = False # Flag to check for the loading of the real PFL module

# --- Centralized System Configuration ---
class SystemConfiguration:
    """Centralized configuration with validation and essential parameters.
    Contains clinical thresholds, scalers for denormalization, etc."""
    NB_FEATURES: Optional[int] = None # Number of features, defined after data loading
    NB_RULES: int = 7 # Number of fuzzy rules (clusters)
    EPSILON: float = 1e-8 # Small value for numerical stability

    # Fixed clinical thresholds for global risk interpretation (not directly used by the simulator, but useful)
    CLINICAL_RISK_THRESHOLDS: Dict[str, float] = {
        'LOW': 10.0,
        'MEDIUM': 15.0,
        'HIGH': 25.0
    }

    # Dictionary to store scalers and raw data statistics
    SCALERS: Dict[str, Any] = {
        'min_max_transformed': {}, # Old scaler for transformed data, potentially unused here
        'raw_stats': {}, # Statistics (min, max, mean, quartiles) of raw features
        'feature_scaler_obj': None, # The MinMaxScaler object trained on raw data
        'feature_scaler_features': [] # Names of features in the order used by the scaler
    }

    @classmethod
    def validate_configuration(cls):
        """Validates configuration consistency."""
        if cls.NB_FEATURES is None or cls.NB_RULES is None:
            logger.warning("NB_FEATURES or NB_RULES not defined in the configuration.")
        logger.info(" ✓ System configuration validated.")

# --- Data Management ---

class SimulatorDataManager:
    """Manages Drive mounting, locating, and loading the data
    needed for the simulator, including statistics for denormalization."""

    def __init__(self):
        self.mydrive_path: Optional[Path] = None
        self.modeling_data2_path: Optional[Path] = None
        self.all_feature_names: Optional[List[str]] = None
        self.X_train_df: Optional[pd.DataFrame] = None # Transformed data as PFL might expect it
        self.X_train_raw: Optional[pd.DataFrame] = None # Original raw data (for stats and scaler)
        self.y_train_arr: Optional[np.ndarray] = None # Training target
        self.df_train_raw_full: Optional[pd.DataFrame] = None # Full original DataFrame (for column 'A')

    def mount_drive(self):
        """Mounts the Drive into the file system."""
        print("\n" + "=" * 50)
        print("MOUNTING DRIVE")
        print("=" * 50)
        try:
            # Attempt to import
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
            self.mydrive_path = Path("/content/drive/MyDrive")
            print(" ✓ Drive mounted successfully.")
        except ImportError:
            # Handling for other environments (e.g., local execution)
            logger.error("Error: Drive not found. Execution outside of Drive detected.")
            # WARNING: For local execution, you MUST adjust this path manually
            # self.mydrive_path = Path("/path/to/your/MyDrive/folder")
            self.mydrive_path = Path("./") # Default to current directory, change if needed
            print(f" ⚠️ Using a default local path: {self.mydrive_path}. Please modify if necessary.")
        except Exception as e:
            logger.error(f"Unexpected failure to mount Drive: {e}")
            raise

    def locate_data_folder(self):
        """Locates the 'modeling_data2' folder in the Drive (or local path)."""
        print("\n" + "=" * 50)
        print("LOCATING DATA FOLDER")
        print("=" * 50)
        # Try the direct path first
        pfl_dir_candidate = self.mydrive_path / "PFL"
        if pfl_dir_candidate.is_dir() and (pfl_dir_candidate / "modeling_data2").is_dir():
            self.modeling_data2_path = pfl_dir_candidate / "modeling_data2"
            print(f" ✓ modeling_data2 folder found: {self.modeling_data2_path}")
            return

        # Fallback: general search if the direct path fails
        print("General search for 'modeling_data2' folder...")
        for root, dirs, _ in os.walk(self.mydrive_path):
            if "modeling_data2" in dirs:
                self.modeling_data2_path = Path(root) / "modeling_data2"
                print(f" ✓ modeling_data2 folder found: {self.modeling_data2_path}")
                return
        raise FileNotFoundError("The 'modeling_data2' folder was not found in the specified Drive path.")

    def load_data(self):
        """Loads training data (df_train, X_train, y_train) from the folder,
        and prepares the scalers for denormalization."""
        print("\n" + "=" * 50)
        print("LOADING DATA AND PREPARING SCALERS")
        print("=" * 50)

        try:
            # Load the full DataFrame for the 'A' column and original feature names
            df_train_path = self.modeling_data2_path / 'df_train.hdf'
            if not df_train_path.exists():
                raise FileNotFoundError(f"File not found: {df_train_path}")
            self.df_train_raw_full = pd.read_hdf(df_train_path, key='Processed_data')

            # Load X_train as a basis for model features (without 'A')
            X_train_temp_df = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data')
            self.all_feature_names = [col for col in X_train_temp_df.columns if col != 'A']
            SystemConfiguration.NB_FEATURES = len(self.all_feature_names)

            # Prepare X_train_df for PFL model training (which will expect these features)
            self.X_train_df = X_train_temp_df.copy()
            # Ensure column 'A' is present if it needs to be in X_train_df for segmentation
            if 'A' in self.df_train_raw_full.columns and 'A' not in self.X_train_df.columns:
                self.X_train_df['A'] = self.df_train_raw_full['A']


            # Load X_train_raw specifically for statistics and the MinMaxScaler
            # This is the basis for future denormalizations and default values for hypothetical patients
            self.X_train_raw = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data').copy()
            # Ensure column 'A' is present if it needs to be in X_train_raw for means
            if 'A' in self.df_train_raw_full.columns and 'A' not in self.X_train_raw.columns:
                self.X_train_raw['A'] = self.df_train_raw_full['A']

            self.y_train_arr = pd.read_hdf(self.modeling_data2_path / 'y_train.hdf', key='Processed_data').values.ravel()

            # --- Calculate and store statistics of ORIGINAL RAW data for denormalization ---
            print("\nCalculating and storing statistics of real values (Min/Max/Mean/Quartiles)...")
            features_for_scaler_and_stats = [col for col in self.X_train_raw.columns if col != 'A']
            X_for_scaler_fit = self.X_train_raw[features_for_scaler_and_stats].select_dtypes(include=np.number)

            if X_for_scaler_fit.isnull().any().any():
                logger.warning("NaNs were found in the raw data; they will be imputed with the mean for the MinMaxScaler.")
                X_for_scaler_fit = X_for_scaler_fit.fillna(X_for_scaler_fit.mean())

            scaler = MinMaxScaler()
            scaler.fit(X_for_scaler_fit) # Train the scaler on the numeric raw data
            SystemConfiguration.SCALERS['feature_scaler_obj'] = scaler
            SystemConfiguration.SCALERS['feature_scaler_features'] = X_for_scaler_fit.columns.tolist()

            raw_stats_dict = {}
            for feature_name in features_for_scaler_and_stats:
                if feature_name in self.X_train_raw.columns and pd.api.types.is_numeric_dtype(self.X_train_raw[feature_name]):
                    col_data = self.X_train_raw[feature_name].dropna()
                    if not col_data.empty:
                        raw_stats_dict[feature_name] = {
                            'min': col_data.min(),
                            'max': col_data.max(),
                            'mean': col_data.mean(),
                            '25th_percentile': col_data.quantile(0.25),
                            '50th_percentile': col_data.quantile(0.50),
                            '75th_percentile': col_data.quantile(0.75)
                        }
                    else:
                        logger.warning(f"Feature '{feature_name}' is empty after NaN removal, statistics not available.")
                else:
                    logger.warning(f"Feature '{feature_name}' is not numeric or was not found in the raw data for statistics.")
            SystemConfiguration.SCALERS['raw_stats'] = raw_stats_dict

            print(" ✓ Data loaded and scalers prepared successfully!")

        except FileNotFoundError as e:
            logger.error(f"Error: Data file not found: {e}")
            raise
        except Exception as e:
            logger.error(f"Unexpected error during data loading or scaler preparation: {e}")
            traceback.print_exc()
            raise

# --- Dynamic PFL Module Loader ---

class SimulatorPFLZModuleLoader:
    """Manages the dynamic loading of the PFL class
    from its external file, ensuring it is correctly imported."""

    @staticmethod
    def load_pflz_cp_module(mydrive_path: Path):
        """Loads the PFL module and returns the aliased real PFL class."""
        print("\n" + "=" * 50)
        print("LOADING PFL MODULE")
        print("=" * 50)
        pflz_file_path = None
        # Try to find the PFLZ.py file
        candidate_path = mydrive_path / "PFL" / "Modules" / "PFLZ.py"
        if candidate_path.exists():
            pflz_file_path = candidate_path
        else:
            # Fallback: broader search if the direct path fails
            for root, dirs, files in os.walk(mydrive_path):
                if "PFL" in dirs:
                    pfl_dir = Path(root) / "PFL"
                    if "Modules" in os.listdir(pfl_dir):
                        modules_path = pfl_dir / "Modules"
                        if "PFLZ.py" in os.listdir(modules_path):
                            pflz_file_path = modules_path / "PFLZ.py"
                            break

        if pflz_file_path is None:
            raise FileNotFoundError("PFL.py file not found. Please ensure PFL.py exists in the 'PFL/Modules' folder.")
        print(f" ✓ PFL.py file found: {pflz_file_path}")

        # Load the module using importlib.util for fine-grained path control
        spec = importlib.util.spec_from_file_location("PFL_module_temp_simulateur", pflz_file_path)
        if spec is None:
            raise ImportError(f"Could not specify module for {pflz_file_path}.")

        module = importlib.util.module_from_spec(spec)
        sys.modules["PFLZ_module_temp_simulateur"] = module # Add the module to sys.modules
        spec.loader.exec_module(module) # Execute the module's code

        # Retrieve the 'PFL' class from the loaded module
        loaded_pflz_class = getattr(module, 'PFLZ', None)
        if loaded_pflz_class is None:
            raise ImportError("The class 'PFLZ' could not be found in the loaded module. Check the class name in PFLZ.py.")

        # Create a PFL_CP_Alias that inherits from the dynamically loaded PFLZ class.
        # This allows the rest of the pipeline to use PFL_CP transparently.
        class PFLZ_CP_Alias(loaded_pflz_class):
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
                # Ensure the existence of X_original_fit, useful for plotting or denormalization
                if not hasattr(self, 'X_original_fit'):
                    self.X_original_fit = None

        print(" ✓ PFL module loaded and aliased successfully.")
        return PFLZ_CP_Alias

# --- Causal Simulator Model ---
class CausalSimulator:
    """Implements the logic for simulating the impact of causal interventions."""

    def __init__(self, pflz_model: Any, all_feature_names: List[str], X_train_raw_for_means: pd.DataFrame):
        """
        Initializes the simulator with the trained PFLZ model and raw training data.
        Args:
            pflz_model: Instance of the trained PFLZ_CP model.
            all_feature_names: List of all feature names used by the PFL model.
            X_train_raw_for_means: DataFrame of raw training data (without 'A', used for means).
        """
        self.pflz_model = pflz_model
        self.all_feature_names = all_feature_names
        # Calculate the means of the numeric features of the raw training data, without the 'A' column
        self.X_train_raw_means = X_train_raw_for_means.drop('A', axis=1, errors='ignore').mean(numeric_only=True)

        # Retrieve scalers and raw statistics from the system configuration
        self.feature_scaler_obj = SystemConfiguration.SCALERS.get('feature_scaler_obj')
        self.feature_scaler_features = SystemConfiguration.SCALERS.get('feature_scaler_features')
        self.raw_stats = SystemConfiguration.SCALERS.get('raw_stats')

        # Critical check for scaler availability
        if self.feature_scaler_obj is None or not self.feature_scaler_features or not self.raw_stats:
            logger.error("Critical error: Scalers or raw statistics are missing. The simulator cannot function correctly for denormalization.")
            raise ValueError("Missing scaler configuration for the causal simulator.")

    def _prepare_patient_instance(self, user_input_values: Dict[str, float]) -> np.ndarray:
        """
        Prepares a normalized feature vector for a given patient instance.
        Features not specified by the user are filled with training means.
        Args:
            user_input_values (Dict[str, float]): Dictionary of user-specified features and their raw values.
        Returns:
            np.ndarray: The normalized feature vector (1D) for inference by the PFL model.
        """
        # Create a DataFrame row, initialized with raw training means
        # Ensure the columns are in the order expected by the scaler
        patient_data_raw = self.X_train_raw_means.to_dict()

        # Apply user-specified values, overwriting the default means
        for feature, value in user_input_values.items():
            if feature in patient_data_raw:
                patient_data_raw[feature] = value
            else:
                logger.warning(f"User-specified feature '{feature}' is not a known training feature. Ignored.")

        # Convert to DataFrame for manipulation and ordered selection by the scaler
        patient_df_raw = pd.DataFrame([patient_data_raw])

        # Select and order columns according to the order expected by the MinMaxScaler
        # and ensure they are numeric
        X_patient_ordered_for_transform = patient_df_raw[self.feature_scaler_features].select_dtypes(include=np.number)

        # Fill NaNs if, for example, a feature was not numeric in the means
        if X_patient_ordered_for_transform.isnull().any().any():
            logger.warning("NaNs detected in the patient instance before normalization; imputed with the feature's mean.")
            X_patient_ordered_for_transform = X_patient_ordered_for_transform.fillna(self.X_train_raw_means[X_patient_ordered_for_transform.columns])

        # Normalize the values using the scaler trained on the raw data
        # reshape(-1, 1) or (1, -1) is necessary for transform to expect a 2D input
        X_patient_normalized_values = self.feature_scaler_obj.transform(X_patient_ordered_for_transform.values)

        return X_patient_normalized_values[0] # Returns a 1D array

    def simulate_intervention_impact(self, patient_original_values: Dict[str, float],
                                      intervention: Dict[str, float]) -> None:
        """
        Simulates the impact of an intervention on a hypothetical patient and displays the changes.
        Args:
            patient_original_values (Dict[str, float]): Initial patient values (user-specified features).
            intervention (Dict[str, float]): Dictionary of features to modify and their new (target) values.
        """
        print("\n" + "=" * 70)
        print("CAUSAL INTERVENTION IMPACT SIMULATION")
        print("=" * 70)

        # --- Initial Patient State (Before Intervention) ---
        print("\n--- Initial Patient State Analysis ---")
        patient_initial_norm = self._prepare_patient_instance(patient_original_values).reshape(1, -1) # Reshape for predict_proba

        # Predict mortality probabilities with the PFL model
        initial_proba_all_classes = self.pflz_model.predict_proba(patient_initial_norm)
        initial_mort_proba = initial_proba_all_classes[0, 1] # Probability of class 1 (mortality)

        # Identify the initial dominant cluster and retrieve its rule
        initial_fuzzy_memberships = self.pflz_model.calc_mf(patient_initial_norm, normalized=True)
        initial_dominant_cluster_idx = np.argmax(initial_fuzzy_memberships[0])

        initial_cluster_explanations = self._generate_interpretable_rule_for_cluster(initial_dominant_cluster_idx)

        print(f"Predicted mortality probability (initial): {initial_mort_proba:.2%}")
        print(f"Dominant risk cluster (initial): Cluster {initial_dominant_cluster_idx}")
        print(f"Rule associated with Cluster {initial_dominant_cluster_idx} (initial):")
        print(f"   P(mortality|cluster) = {initial_cluster_explanations['conditional_prob']:.2%}")
        print(f"   Rule: {initial_cluster_explanations['rule']}")

        # --- Patient State After Intervention ---
        print("\n--- Patient State After Intervention (displaying changes) ---")

        # Build the patient's values after the intervention
        patient_after_intervention_values = patient_original_values.copy()
        for feature, new_value in intervention.items():
            if feature in self.all_feature_names:
                patient_after_intervention_values[feature] = new_value
            else:
                logger.warning(f"Feature '{feature}' specified in the intervention is not a known training feature. Ignored.")

        patient_after_norm = self._prepare_patient_instance(patient_after_intervention_values).reshape(1, -1)

        # New prediction and dominant cluster identification
        after_proba_all_classes = self.pflz_model.predict_proba(patient_after_norm)
        after_mort_proba = after_proba_all_classes[0, 1]

        after_fuzzy_memberships = self.pflz_model.calc_mf(patient_after_norm, normalized=True)
        after_dominant_cluster_idx = np.argmax(after_fuzzy_memberships[0])

        after_cluster_explanations = self._generate_interpretable_rule_for_cluster(after_dominant_cluster_idx)

        print(f"Predicted mortality probability (before): {initial_mort_proba:.2%}")
        print(f"Predicted mortality probability (after): {after_mort_proba:.2%}")

        # Display only the change in the rule
        if initial_dominant_cluster_idx == after_dominant_cluster_idx:
            print(f"The patient remains in the same risk profile (Cluster {initial_dominant_cluster_idx}).")
            print("The feature modifications led to a change in probability but did not change the dominant risk profile.")
            # Optional: here we could generate the rule for the "after" cluster and compare the intervals that changed.
            # For now, we simply state that the cluster rule is the same.
        else:
            print(f"The patient's risk profile has changed:")
            print(f"   - Old dominant Cluster: Cluster {initial_dominant_cluster_idx}")
            print(f"   - New dominant Cluster: Cluster {after_dominant_cluster_idx}")
            print(f"The inherent probability of the new cluster is {after_cluster_explanations['conditional_prob']:.2%}.")
            print(f"The new dominant risk rule is:")
            print(f"   Rule: {after_cluster_explanations['rule']}")

        # --- Comparative Summary of Impact ---
        print("\n--- Summary of the intervention's impact ---")
        proba_delta = after_mort_proba - initial_mort_proba
        if proba_delta < -0.05:
            print(f"The simulated intervention appears to **significantly reduce** the probability of mortality ({proba_delta:.2%} reduction).")
        elif proba_delta > 0.05:
            print(f"The simulated intervention appears to **increase** the probability of mortality ({proba_delta:.2%} increase), which could be an undesirable or non-optimal effect.")
        else:
            print(f"The simulated intervention has a **marginal impact** on the probability of mortality ({proba_delta:.2%} change).")

        print("\n" + "=" * 70)
        print("SIMULATION COMPLETED")
        print("=" * 70)

    def _generate_interpretable_rule_for_cluster(self, cluster_id: int, alpha: float = 1.5) -> Dict:
        """
        Generates the readable clinical rule for a given cluster, with denormalization and linguistic qualifiers.
        Args:
            cluster_id (int): The cluster identifier.
            alpha (float): Adjustment factor for the interval width.
        Returns:
            Dict: Dictionary containing the conditional probability and the denormalized rule.
        """
        if self.pflz_model.cluster_centers_ is None or self.pflz_model.cluster_centers_.shape[0] == 0 or \
           self.pflz_model.feature_names is None or not hasattr(self.pflz_model, 'cond_probs_'):
            logger.error(f"Incomplete PFL model to generate rule for cluster {cluster_id}.")
            return {'conditional_prob': np.nan, 'rule': "Rule not available (incomplete PFL model)."}

        if cluster_id >= self.pflz_model.n_clusters or cluster_id < 0:
            logger.error(f"Cluster ID {cluster_id} is out of model limits ({self.pflz_model.n_clusters} clusters).")
            return {'conditional_prob': np.nan, 'rule': "Invalid Cluster ID."}

        cluster_center_norm = self.pflz_model.cluster_centers_[cluster_id]

        # Retrieve rule widths (sigmas) from the PFL model
        rule_widths_all = None
        try:
            rule_widths_all = self.pflz_model.calc_rule_widths()
        except Exception as e:
            logger.warning(f"Error calling pflz_model.calc_rule_widths(): {e}. Using default widths.")
            # Fallback to a standard width if the method fails
            rule_widths_all = np.full((self.pflz_model.n_clusters, self.pflz_model.n_original_features_), 0.5)

        rule_widths = rule_widths_all[cluster_id] if rule_widths_all is not None and rule_widths_all.shape[0] > cluster_id else None

        if rule_widths is None or len(rule_widths) != len(self.pflz_model.feature_names):
            logger.warning("Rule widths not available or incorrect for the cluster. Using default widths.")
            rule_widths = np.full(len(self.pflz_model.feature_names), 0.5) # Fallback for width

        conditions = []

        # Prepare the vector for the inverse transformation of the centroid, in the order of the scaler
        centroid_for_inverse_transform = np.zeros(len(self.feature_scaler_features))
        feature_name_to_scaler_idx = {name: i for i, name in enumerate(self.feature_scaler_features)}

        for i, feature_name_model in enumerate(self.pflz_model.feature_names):
            if feature_name_model in feature_name_to_scaler_idx:
                scaler_idx = feature_name_to_scaler_idx[feature_name_model]
                centroid_for_inverse_transform[scaler_idx] = cluster_center_norm[i]
            # PFLZ model features that are not in the scaler_features_order will remain at 0 in this vector.

        # Denormalize the cluster center
        denormalized_center_full = None
        if self.feature_scaler_obj is not None:
            try:
                denormalized_center_full = self.feature_scaler_obj.inverse_transform(centroid_for_inverse_transform.reshape(1, -1))[0]
            except Exception as e:
                logger.error(f"Error during cluster center denormalization: {e}. Center values will be NaN.")
                denormalized_center_full = np.full(len(self.feature_scaler_features), np.nan)
        else:
            logger.error("Feature scaler not found for denormalization. Rules will be displayed in internal normalized values.")


        # Build the denormalized conditions for each feature of the rule
        for i, feature_name_model in enumerate(self.pflz_model.feature_names):
            # Check if raw statistics and position in the scaler are available for the feature
            if feature_name_model not in self.raw_stats or feature_name_model not in feature_name_to_scaler_idx:
                # If stats or mapping to the scaler are missing, fall back to internal normalized display
                normalized_value = cluster_center_norm[i]
                feature_width = rule_widths[i] if i < len(rule_widths) else 0.5
                lower_bound_norm = normalized_value - alpha * feature_width
                upper_bound_norm = normalized_value + alpha * feature_width
                conditions.append(f"{feature_name_model} ∈ [{lower_bound_norm:.3f}, {upper_bound_norm:.3f}] (Internal Normalized - Raw stats missing)")
                continue

            # Get the raw statistics and the denormalized center value for this feature
            raw_stats_feat = self.raw_stats[feature_name_model]
            raw_min = raw_stats_feat['min']
            raw_max = raw_stats_feat['max']
            raw_range = raw_max - raw_min

            scaler_idx = feature_name_to_scaler_idx[feature_name_model]
            denormalized_val_center = denormalized_center_full[scaler_idx] # Denormalized center value

            # Calculate the interval width in the denormalized space
            interval_half_width_norm = alpha * (rule_widths[i] if i < len(rule_widths) else 0.5)
            # To convert the normalized width to denormalized width, it must be multiplied by the real range
            denormalized_interval_half_width = interval_half_width_norm * raw_range

            lower_bound_denorm = denormalized_val_center - denormalized_interval_half_width
            upper_bound_denorm = denormalized_val_center + denormalized_interval_half_width

            # --- Add linguistic qualifiers ---
            position_interpretation = ""
            # Use percentiles of the raw data for position qualifiers
            p25 = raw_stats_feat.get('25th_percentile', np.nan)
            p50 = raw_stats_feat.get('50th_percentile', np.nan)
            p75 = raw_stats_feat.get('75th_percentile', np.nan)

            if not np.isnan(p25) and not np.isnan(p75):
                center_of_denorm_interval = (lower_bound_denorm + upper_bound_denorm) / 2
                if center_of_denorm_interval < p25:
                    position_interpretation = "rather low"
                elif center_of_denorm_interval > p75:
                    position_interpretation = "rather high"
                else:
                    position_interpretation = "average"

            width_interpretation = ""
            if raw_range > SystemConfiguration.EPSILON:
                rule_range_denorm = upper_bound_denorm - lower_bound_denorm
                ratio_width = rule_range_denorm / raw_range
                if ratio_width < 0.10: # If the rule covers less than 10% of the feature's real range
                    width_interpretation = "very precise"
                elif ratio_width < 0.30: # If the rule covers between 10% and 30%
                    width_interpretation = "quite precise"
                else: # If the rule covers more than 30%
                    width_interpretation = "wide"
            else:
                width_interpretation = "constant" # For features with a range of 0 (or very small)

            conditions.append(f"{feature_name_model} ∈ [{lower_bound_denorm:.2f}, {upper_bound_denorm:.2f}] (value {position_interpretation}, range {width_interpretation})")

        # Replacements of feature names for better clinical readability
        # The `replacements` dictionary is cleared to avoid translating variable names
        replacements = {}
        simplified_rule = " AND ".join(conditions)
        for old, new in replacements.items():
            simplified_rule = simplified_rule.replace(old, new)

        # Conditional probability P(mortality|cluster)
        cond_prob = self.pflz_model.cond_probs_[cluster_id, 1] if self.pflz_model.cond_probs_ is not None and self.pflz_model.cond_probs_.shape[1] > 1 else np.nan

        return {
            'conditional_prob': cond_prob,
            'rule': simplified_rule
        }

# --- Main Pipeline for Simulator Execution ---
class CausalSimulatorPipeline:
    """Orchestrates model loading, training, and the execution of the causal simulation."""

    def __init__(self):
        self.data_manager = SimulatorDataManager()
        self.pflz_model_instance: Any = None # The instance of the trained PFL model
        self.all_feature_names: Optional[List[str]] = None
        self.X_train_raw_data: Optional[pd.DataFrame] = None # The full raw data for means

    def _initialize_and_load_all(self):
        """Initialization and loading of all data and the PFL module."""
        print("\n" + "=" * 60)
        print("INITIALIZATION AND SYSTEM LOADING FOR SIMULATOR")
        print("=" * 60)
        self.data_manager.mount_drive()
        self.data_manager.locate_data_folder()
        self.data_manager.load_data()

        # Update pipeline attributes from the data manager
        self.all_feature_names = self.data_manager.all_feature_names
        self.X_train_raw_data = self.data_manager.X_train_raw # Store the full raw DataFrame

        global PFLZ_CP, _PFLZ_MODULE_LOADED # Declare the intention to modify global variables

        try:
            loaded_pflz_class = SimulatorPFLZModuleLoader.load_pflz_cp_module(self.data_manager.mydrive_path)
            if loaded_pflz_class is None:
                raise ImportError("The PFL class could not be found or loaded.")
            PFLZ_CP = loaded_pflz_class # Overwrite the global placeholder with the real class
            _PFLZ_MODULE_LOADED = True
            logger.info(" ✓ PFL module loaded and validated for simulator execution.")
        except Exception as e:
            logger.error(f" ❌ Error loading PFLZ_CP: {e}")
            _PFLZ_MODULE_LOADED = False
            raise

    def _train_pflz_model(self, subset_key: str):
        """
        Trains the PFL model on the data subset chosen by the user.
        Args:
            subset_key (str): Key for the subset ('A0', 'A1', 'ALL').
        """
        print("\n" + "=" * 60)
        print(f"TRAINING PFL MODEL FOR SUBSET: {subset_key}")
        print("=" * 60)

        if not _PFLZ_MODULE_LOADED:
            raise RuntimeError("The PFL module was not loaded correctly. Cannot train the model.")

        # Prepare data subsets for PFL training
        # The PFL expects features WITHOUT the 'A' column for training.
        X_train_no_A_for_pflz = self.data_manager.X_train_df.drop('A', axis=1, errors='ignore')

        X_data_subset, y_data_subset = None, None
        if subset_key == 'A0':
            mask = self.data_manager.X_train_df['A'] == 0
            X_data_subset = X_train_no_A_for_pflz[mask]
            y_data_subset = self.data_manager.y_train_arr[mask]
        elif subset_key == 'A1':
            mask = self.data_manager.X_train_df['A'] == 1
            X_data_subset = X_train_no_A_for_pflz[mask]
            y_data_subset = self.data_manager.y_train_arr[mask]
        elif subset_key == 'ALL':
            X_data_subset = X_train_no_A_for_pflz
            y_data_subset = self.data_manager.y_train_arr
        else:
            raise ValueError(f"Unknown subset: {subset_key}")

        if X_data_subset.empty or y_data_subset.size == 0:
            logger.warning(f"Insufficient data to train the model on subset {subset_key}. Training is skipped.")
            return

        # LightGBM parameters (these parameters should ideally come from SystemConfiguration,
        # but are defined here to show they are passed to PFL_CP)
        lgb_params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'verbose': -1, # Suppress LightGBM training log display
            'random_state': 42,
            'force_col_wise': True
        }

        try:
            self.pflz_model_instance = PFLZ_CP(
                n_clusters=SystemConfiguration.NB_RULES,
                epsilon=SystemConfiguration.EPSILON,
                lgb_params=lgb_params,
                feature_names=self.all_feature_names # Feature names passed to PFLZ for its interpretability
            )
            print(f"Starting PFL model training for subset {subset_key}...")
            # PFLZ.fit expects a numpy array, so convert from DataFrame
            self.pflz_model_instance.fit(X_data_subset.values, y_data_subset)
            self.pflz_model_instance.X_original_fit = X_data_subset.values # Important for some internal PFLZ functions
            print(" ✓ PFL model trained successfully!")
        except Exception as e:
            logger.error(f"Error during PFLZ model training: {e}")
            raise

    def _collect_patient_inputs(self) -> Dict[str, float]:
        """
        Collects hypothetical patient feature values from the user.
        Displays real ranges to guide input.
        Returns:
            Dict[str, float]: Dictionary of user-specified features and their values.
        """
        print("\n" + "=" * 70)
        print("INPUT OF HYPOTHETICAL PATIENT FEATURES (INITIAL STATE)")
        print("=" * 70)
        print("\nHere is the list of available features for specification, with their real ranges (Min/Max) and quartiles:")
        raw_stats = SystemConfiguration.SCALERS.get('raw_stats', {})
        for i, feature_name in enumerate(self.all_feature_names):
            if feature_name in raw_stats:
                stats = raw_stats[feature_name]
                print(f"    {i + 1}. {feature_name}: Min={stats['min']:.2f}, Max={stats['max']:.2f}, Mean={stats['mean']:.2f}, Q1={stats['25th_percentile']:.2f}, Median={stats['50th_percentile']:.2f}, Q3={stats['75th_percentile']:.2f}")
            else:
                print(f"    {i + 1}. {feature_name}: Real statistics not available (non-numeric or outside of scaler stats).")

        user_input_values = {}
        num_features_to_spec = -1
        MAX_FEATURES_TO_SPEC = len(self.all_feature_names)

        while num_features_to_spec < 0 or num_features_to_spec > MAX_FEATURES_TO_SPEC:
            try:
                num_features_to_spec_str = input(f"\nHow many features do you want to specify (0-{MAX_FEATURES_TO_SPEC})? ").strip()
                num_features_to_spec = int(num_features_to_spec_str)
            except ValueError:
                print("Please enter a valid integer.")

        if num_features_to_spec > 0:
            chosen_indices = []
            valid_indices_entered = False
            while not valid_indices_entered:
                try:
                    feature_numbers_str = input(f"Enter the NUMBERS of the features (separated by commas, e.g., 1, 14): ").strip()
                    input_parts = [part.strip() for part in feature_numbers_str.split(',') if part.strip()]
                    temp_indices = []
                    all_parts_valid = True
                    for part in input_parts:
                        if part.isdigit():
                            idx = int(part) - 1
                            if 0 <= idx < len(self.all_feature_names):
                                if idx not in temp_indices:
                                    temp_indices.append(idx)
                                else:
                                    print(f"Warning: Feature {self.all_feature_names[idx]} (number {int(part)}) was entered twice and will be ignored.")
                            else:
                                print(f"Invalid feature number: {int(part)}. Please enter all numbers again.")
                                all_parts_valid = False
                                break
                        else:
                            print(f"Invalid format for '{part}'. Please enter only integers separated by commas.")
                            all_parts_valid = False
                            break
                    if all_parts_valid:
                        if len(temp_indices) != num_features_to_spec:
                            print(f"You asked to specify {num_features_to_spec} features, but you entered {len(temp_indices)}. Please try again.")
                        else:
                            chosen_indices = temp_indices
                            valid_indices_entered = True
                except Exception as e:
                    print(f"Error reading feature numbers: {e}. Please try again.")

            for idx in chosen_indices:
                chosen_feature_name = self.all_feature_names[idx]
                valid_value = False
                while not valid_value:
                    try:
                        value = float(input(f"Enter the value for '{chosen_feature_name}': "))
                        # Warning if outside of real range
                        if chosen_feature_name in raw_stats:
                            min_v = raw_stats[chosen_feature_name]['min']
                            max_v = raw_stats[chosen_feature_name]['max']
                            if not (min_v <= value <= max_v):
                                print(f"Warning: The value ({value:.2f}) is outside the real range ({min_v:.2f}-{max_v:.2f}) observed in your training data for '{chosen_feature_name}'.")
                        user_input_values[chosen_feature_name] = value
                        valid_value = True
                    except ValueError:
                        print("Please enter a valid numeric value.")
        return user_input_values

    def _collect_intervention_inputs(self) -> Dict[str, float]:
        """
        Collects the features to modify for the intervention simulation.
        Displays real statistics to guide the input of target values.
        Returns:
            Dict[str, float]: Dictionary of features to modify and their new target values.
        """
        print("\n" + "=" * 70)
        print("DEFINITION OF SIMULATED INTERVENTION (TARGETED CHANGES)")
        print("=" * 70)
        print("Please specify the features you want to modify to simulate an intervention.")
        print("Refer to the real statistics (Min/Max/Mean/Quartiles) displayed previously for realistic target values.")

        intervention_values = {}
        features_to_modify_count = -1

        while features_to_modify_count < 0 or features_to_modify_count > len(self.all_feature_names):
            try:
                features_to_modify_count_str = input(f"\nHow many features do you want to modify for the simulation (0-{len(self.all_feature_names)})? ").strip()
                features_to_modify_count = int(features_to_modify_count_str)
            except ValueError:
                print("Please enter a valid integer.")

        if features_to_modify_count > 0:
            chosen_features_for_intervention_names = []
            while len(chosen_features_for_intervention_names) < features_to_modify_count:
                try:
                    # Display available features if the user needs them, but not every time
                    if len(chosen_features_for_intervention_names) == 0:
                        print("\nAvailable features for modification (with numbers):")
                        for i, feat_name in enumerate(self.all_feature_names):
                            print(f"    {i+1}. {feat_name}")

                    feat_num_str = input(f"Enter the NUMBER of the feature to modify (remaining {features_to_modify_count - len(chosen_features_for_intervention_names)} to choose): ").strip()
                    feat_idx = int(feat_num_str) - 1

                    if 0 <= feat_idx < len(self.all_feature_names):
                        feature_name = self.all_feature_names[feat_idx]
                        if feature_name not in chosen_features_for_intervention_names:
                            chosen_features_for_intervention_names.append(feature_name)
                            valid_value = False
                            while not valid_value:
                                try:
                                    new_value = float(input(f"Enter the NEW target value for '{feature_name}': "))
                                    # Warning if outside of real range
                                    raw_stats = SystemConfiguration.SCALERS.get('raw_stats', {})
                                    if feature_name in raw_stats:
                                        min_v = raw_stats[feature_name]['min']
                                        max_v = raw_stats[feature_name]['max']
                                        if not (min_v <= new_value <= max_v):
                                            print(f"Warning: The target value ({new_value:.2f}) is outside the real range ({min_v:.2f}-{max_v:.2f}) observed in your training data for '{feature_name}'.")
                                    intervention_values[feature_name] = new_value
                                    valid_value = True
                                except ValueError:
                                    print("Please enter a valid numeric value.")
                        else:
                            print("This feature has already been selected for modification. Please choose another one.")
                    else:
                        print("Invalid feature number. Please enter a number from the list.")
                except ValueError:
                    print("Invalid input. Please enter an integer.")

        return intervention_values

    def execute(self):
        """Executes the complete causal simulation pipeline."""
        try:
            logger.info(" 🚀 Starting the causal simulator pipeline.")
            self._initialize_and_load_all()

            # Ask the user for the data subset for PFL training
            chosen_subset_key = ""
            valid_choices = ['A0', 'A1', 'ALL']
            while chosen_subset_key not in valid_choices:
                print("\n" + "=" * 70)
                print("CHOICE OF TREATMENT GROUP FOR PFL MODEL TRAINING")
                print("=" * 70)
                print("Please choose the antibiotic treatment group to train the PFL model on:")
                print("   - 'A0' for patients on multiple antibiotics")
                print("   - 'A1' for patients on a single antibiotic")
                print("   - 'ALL' for all patients")
                chosen_subset_key = input("Your choice (A0, A1, or ALL): ").strip().upper()
                if chosen_subset_key not in valid_choices:
                    print("Invalid choice. Please enter A0, A1, or ALL.")

            # Train the PFL model on the chosen subset
            self._train_pflz_model(chosen_subset_key)

            # Check if the model was trained successfully
            if self.pflz_model_instance is None:
                raise RuntimeError("The PFL model could not be trained. Cannot continue the simulation.")
            # Ensure the trained model has the essential attributes for simulation
            if not all(hasattr(self.pflz_model_instance, attr) for attr in ['predict_proba', 'calc_mf', 'cluster_centers_', 'cond_probs_', 'feature_names', 'n_clusters', 'calc_rule_widths']):
                 raise RuntimeError("The loaded PFL model instance does not have the expected attributes or methods after training. Check the implementation of PFLZ.py.")


            # Collect initial patient inputs
            patient_initial_values = self._collect_patient_inputs()
            if not patient_initial_values:
                print("\n(No features specified for the initial patient. Simulation ends.)")
                return

            # Collect inputs for the intervention
            intervention_values = self._collect_intervention_inputs()
            if not intervention_values:
                print("\n(No intervention specified. Simulation ends.)")
                return

            # Execute the simulation
            causal_simulator_instance = CausalSimulator(
                pflz_model=self.pflz_model_instance,
                all_feature_names=self.all_feature_names,
                X_train_raw_for_means=self.X_train_raw_data # Pass the full raw DataFrame for means
            )
            causal_simulator_instance.simulate_intervention_impact(
                patient_initial_values, intervention_values
            )

            logger.info(" 🎉 Causal simulator pipeline executed successfully!")

        except Exception as e:
            logger.error(f" 💥 Fatal error in the causal simulator pipeline: {e}")
            traceback.print_exc()
            sys.exit(1)

# --- Main entry point of the script ---
if __name__ == "__main__":
    # Uncomment the line below to disable ALL Python warnings (caution)
    # warnings.filterwarnings('ignore')
    try:
        pipeline = CausalSimulatorPipeline()
        pipeline.execute()
    except Exception as e:
        logger.error(f"Error during main simulator execution: {e}")
        sys.exit(1)


INITIALIZATION AND SYSTEM LOADING FOR SIMULATOR

MOUNTING DRIVE
Mounted at /content/drive
 ✓ Drive mounted successfully.

LOCATING DATA FOLDER
 ✓ modeling_data2 folder found: /content/drive/MyDrive/PFL/modeling_data2

LOADING DATA AND PREPARING SCALERS

Calculating and storing statistics of real values (Min/Max/Mean/Quartiles)...
 ✓ Data loaded and scalers prepared successfully!

LOADING PFL MODULE
 ✓ PFL.py file found: /content/drive/MyDrive/PFL/Modules/PFLZ.py
 ✓ PFL module loaded and aliased successfully.

CHOICE OF TREATMENT GROUP FOR PFL MODEL TRAINING
Please choose the antibiotic treatment group to train the PFL model on:
   - 'A0' for patients on multiple antibiotics
   - 'A1' for patients on a single antibiotic
   - 'ALL' for all patients
Your choice (A0, A1, or ALL): A0

TRAINING PFL MODEL FOR SUBSET: A0
Starting PFL model training for subset A0...
 ✓ PFL model trained successfully!

INPUT OF HYPOTHETICAL PATIENT FEATURES (INITIAL STATE)

Here is the list of available feature


CAUSAL INTERVENTION IMPACT SIMULATION

--- Initial Patient State Analysis ---


Predicted mortality probability (initial): 2.36%
Dominant risk cluster (initial): Cluster 4
Rule associated with Cluster 4 (initial):
   P(mortality|cluster) = 4.83%
   Rule: arterial base excess ∈ [-27.00, 19.30] (value rather low, range wide) AND arterial pco2 ∈ [-30.57, 26.49] (value rather low, range wide) AND arterial ph ∈ [-48.87, 28.35] (value rather low, range wide) AND arterial po2 ∈ [-29.88, 23.51] (value rather low, range wide) AND bicarbonate ∈ [-16.95, 13.37] (value rather low, range wide) AND bilirubin ∈ [-36.09, 25.62] (value rather low, range wide) AND calcium ∈ [-81.14, 51.38] (value rather low, range wide) AND creatinine ∈ [-24.18, 16.35] (value rather low, range wide) AND crp ∈ [-36.52, 4.80] (value rather low, range wide) AND cvp ∈ [-28.17, 16.88] (value rather low, range wide) AND diastolic blood pressure ∈ [-25.50, 18.03] (value rather low, range wide) AND fio2 ∈ [-17.45, 9.24] (value rather low, range wide) AND glucose ∈ [-63.03, 48.61] (value rather low, range w